*This is the notebook for the training phase of this example*

## Feed Forward Network Demonstration
**Goal:** Predict the function $f(x, y) = e^{-(x^2 + y^2)}$

**Strategy:**
1. We build a neural network with architecture $2 - 16 - 16 - 1$. With reLU layer in between.
2. We train with $41 \times 41$ points, from $(-10, -10)$ to $(10, 10)$, with step of $0.5$ in between.
3. We use Adam optimizer to optimize.
4. We save our training result to a `.cache` file.

This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [3]:
import random
import math
import lktorch as lk
import os

CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")

### 2. Generate and preprocess

In [4]:
public_test = []
for i in range(-20, 21):
    for j in range(-20, 21):
        public_test.append([i / 2, j / 2, math.exp(- (i / 2)**2 - (j / 2) ** 2)])

# Preprocess to make the actual training more effective
random.shuffle(public_test) 

print("Done fetching data!")

Done fetching data!


### 3. Training

In [ ]:
public_test_tensor = lk.Tensor(public_test)
input_data = public_test_tensor.Slice([0], [1])
output_data = public_test_tensor.Slice([2], [2])

model = lk.Sequential([
    lk.LinearLayer(2, 16),
    lk.reLU_Layer(),
    lk.LinearLayer(16, 16),
    lk.reLU_Layer(),
    lk.LinearLayer(16, 1),
    lk.Sigmoid_Layer(),
])

optimizer = lk.Adam(0.001, 0.99, 0.99)
optimizer.add_parameter(model.get_parameters())
for epoch in range(1, 10001):
    batch = 101
    l = 0
    while (l < len(public_test)):
        optimizer.zero_gradient()
        
        r = min(l + batch - 1, len(public_test) - 1)
        # batch training
        input_batch = input_data.Slice([l, 0], [r, 1])
        output_batch = output_data.Slice([l, 0], [r, 0])

        tenso = model(input_batch)
        tenso = lk.RMSELoss(tenso, output_batch)
        tenso = lk.MeanAll(tenso)
        tenso.backward()

        l += batch
        optimizer.step()

print("Done training!")

Done training!


### 4. Save to file

In [6]:
lk.WriteFile(model.get_state_dict(), WEIGHT_PATH)
print("File saved successfully!")

File saved successfully!
